In [ ]:
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python

In [ ]:
import base64
import os
from typing import List, Optional, Tuple, Union
from llama_cpp import Llama

In [ ]:
import base64
import os
from typing import List, Optional, Tuple, Union
from llama_cpp import Llama

class LlamaCPPWrapper:
    def __init__(
        self, 
        model_path: str, 
        n_gpu_layers: int = -1,  # -1 означает все слои на GPU (аналог высокой утилизации)
        max_model_len: int = 4096, # Аналог n_ctx
        temperature: float = 0.0,
        max_tokens: int = 1024,
        verbose: bool = False
    ):
        """
        Инициализация модели llama-cpp.
        """
        self.model_path = model_path
        
        # Сохраняем параметры генерации для использования при вызове
        self.generation_kwargs = {
            "temperature": temperature,
            "max_tokens": max_tokens,
        }

        # Инициализируем движок llama.cpp
        # chat_format="chatml" или None (автоопределение) часто подходит для Qwen/Llama
        self.llm = Llama(
            model_path=model_path,
            n_ctx=max_model_len,
            n_gpu_layers=n_gpu_layers,
            verbose=verbose
        )

    def _encode_image_base64(self, image_path: str) -> str:
        """Читает файл и переводит в base64 строку (без изменений)."""
        if not os.path.exists(image_path):
            raise FileNotFoundError(f"Image file not found: {image_path}")
            
        with open(image_path, "rb") as image_file:
            return base64.b64encode(image_file.read()).decode('utf-8')

    def _prepare_messages(self, prompt: str, image_path: Optional[str] = None) -> List[dict]:
        """Формирует структуру сообщений (без изменений)."""
        content = []
        
        # Добавляем текст
        content.append({"type": "text", "text": prompt})
        
        # Добавляем изображение, если путь передан
        if image_path:
            image_b64 = self._encode_image_base64(image_path)
            ext = image_path.split('.')[-1].lower()
            mime = "png" if ext == "png" else "jpeg"
            
            content.append({
                "type": "image_url", 
                "image_url": {"url": f"data:image/{mime};base64,{image_b64}"}
            })

        return [
            {"role": "user", "content": content}
        ]

    def invoke(self, text: str, image_path: Optional[str] = None) -> str:
        """
        Запуск инференса для одного примера.
        """
        messages = self._prepare_messages(text, image_path)
        
        # llama-cpp-python поддерживает API, похожее на OpenAI
        output = self.llm.create_chat_completion(
            messages=messages,
            temperature=self.generation_kwargs["temperature"],
            max_tokens=self.generation_kwargs["max_tokens"]
        )
        
        # Извлечение текста из структуры ответа OpenAI-format
        return output['choices'][0]['message']['content'].strip()

    def invoke_batch(
        self, 
        inputs: List[Tuple[str, Optional[str]]], 
        batch_size: int = 8
    ) -> List[str]:
        """
        Запуск инференса батчами.
        Примечание: В llama-cpp-python батчинг реализован последовательно в цикле,
        так как нативная поддержка batch-decoding отличается от vLLM.
        """
        results = []
        
        # В llama.cpp python bindings проще проходить последовательно, 
        # но сохраним логику функции для совместимости интерфейса.
        for text, img_path in inputs:
            # Вызываем метод invoke для каждого элемента
            # (здесь нет параллельного батчинга как в vLLM, идет последовательная генерация)
            res = self.invoke(text, img_path)
            results.append(res)
            
        return results

if __name__ == "__main__":
    # ВАЖНО: Для llama.cpp нужен путь к файлу .gguf, а не имя репозитория.
    # Например: "models/qwen2.5-0.5b-instruct-q4_k_m.gguf"
    # Если файла нет, код упадет с ошибкой.
    
    # Пример пути (замените на свой реальный путь к GGUF файлу)
    MODEL_PATH = "models/Qwen2.5-0.5B-Instruct-Q4_K_M.gguf" 
    
    if not os.path.exists(MODEL_PATH):
        print(f"Ошибка: Файл модели {MODEL_PATH} не найден. Укажите путь к .gguf файлу.")
    else:
        # Инициализация
        model = LlamaCPPWrapper(model_path=MODEL_PATH)

        # 1. Одиночный запуск (только текст)
        print("--- Single Text ---")
        res1 = model.invoke("Сколько будет 2+2?")
        print("Answer:", res1)

        # 2. Одиночный запуск (текст + картинка)
        # Примечание: Для работы с картинками модель должна быть мультимодальной (например, LLaVA или Qwen-VL в GGUF)
        # И должен быть правильно настроен chat_handler или projector, если используется старый API.
        # Современный llama-cpp-python умеет распознавать image_url для LLaVA моделей автоматически.
        # print("--- Single Image ---")
        # res2 = model.invoke("Describe this image", image_path="test.jpg")
        # print("Answer:", res2)

        # 3. Пакетная обработка
        print("\n--- Batch Processing ---")
        batch_data = [
            ("What is capital of France?", None),
            ("Explain quantum physics in one sentence", None),
        ]
        
        batch_results = model.invoke_batch(batch_data, batch_size=2)
        for q, a in zip(batch_data, batch_results):
            print(f"Q: {q[0]}\nA: {a}\n")

In [1]:
!pip install llama-cpp-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 MB 2.8 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.3.16-cp311-cp311-macosx_26_0_arm64.whl size=3847304 sha256=cbff53480b714507d023e064efe8979df24ebd7daf336e569b2e389b44916ad8
  Stored in directory: /Users/artemdzalilov/Library/Caches/pip/wheels/d8/5b/e5/a7d4b5765da347d314e8155197440c9995a962f8e4a5f52b23
Successfully built llama-cpp-python

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
